# 01 From Scratch — Scalar Autograd

**Status:** Complete

**Learning outcome:** Build a micrograd-style scalar autograd engine from scratch — understanding computation graphs, topological sort, and reverse-mode autodiff at the atomic level.

## Why This Matters

Every modern deep learning framework (PyTorch, JAX, TensorFlow) relies on automatic differentiation. Understanding how gradients flow backward through a computation graph is fundamental to:
- Debugging training failures (vanishing/exploding gradients)
- Designing custom loss functions
- Understanding why certain architectures train better than others

We build a scalar autograd engine (similar to Karpathy's micrograd) that supports addition, multiplication, power, tanh, and exp — enough to construct and train a small neural network.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import math

np.random.seed(42)
print("Libraries loaded.")

Libraries loaded.


## Theory Sketch — Reverse-Mode Autodiff

Given a computation graph where output $L$ depends on inputs $x_1, \ldots, x_n$ through intermediate nodes:

1. **Forward pass**: Compute each node's value in topological order
2. **Backward pass**: Starting from $\frac{\partial L}{\partial L} = 1$, propagate gradients backward using the chain rule:
   - For $c = a + b$: $\frac{\partial L}{\partial a} \mathrel{+}= \frac{\partial L}{\partial c} \cdot 1$, same for $b$
   - For $c = a \cdot b$: $\frac{\partial L}{\partial a} \mathrel{+}= \frac{\partial L}{\partial c} \cdot b$, and $\frac{\partial L}{\partial b} \mathrel{+}= \frac{\partial L}{\partial c} \cdot a$
   - For $c = \tanh(a)$: $\frac{\partial L}{\partial a} \mathrel{+}= \frac{\partial L}{\partial c} \cdot (1 - c^2)$

Key insight: **+=** not **=** because a node may be used multiple times (multivariate chain rule).

## The Value Class — Core Autograd Engine

In [2]:
class Value:
    """Scalar value with autograd support."""

    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only int/float powers"
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        e = math.exp(self.data)
        out = Value(e, (self,), 'exp')
        def _backward():
            self.grad += e * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * (other ** -1) if isinstance(other, Value) else self * (Value(other) ** -1)

print("Value class defined.")

# Quick smoke test
a = Value(2.0, label='a')
b = Value(3.0, label='b')
c = a * b + a
print(f"a*b + a = {c.data}  (expected 8.0)")
assert c.data == 8.0

Value class defined.
a*b + a = 8.0  (expected 8.0)


---
**Understanding Computation Graphs**

**Plain language:** Think of a computation graph like a recipe card. Each ingredient is a starting node, each cooking step (mix, heat, chop) is an operation, and arrows show which ingredients flow into which steps. If you want to know "how much would the final dish change if I used a little more salt?", you need to trace salt's influence through every step it touched. That tracing is exactly what automatic differentiation does.

**Building intuition:** In a computation graph, every intermediate quantity gets its own node, and directed edges connect inputs to outputs through operations like addition and multiplication. The graph records not just *what* was computed but *how* — which parent values produced each result. This structural record is what lets us mechanically apply the chain rule: we know exactly which partial derivatives to multiply along each path from output back to input.

**Formal statement:** A computation graph is a directed acyclic graph (DAG) $G = (V, E)$ where each node $v_i \in V$ stores a scalar value and each edge $(v_j, v_i) \in E$ indicates that $v_j$ is a direct input to the operation producing $v_i$. The forward pass evaluates nodes in topological order. The backward pass computes $\frac{\partial L}{\partial v_i}$ for every node by traversing the DAG in reverse topological order, applying the chain rule: $\frac{\partial L}{\partial v_j} = \sum_{v_i \in \text{children}(v_j)} \frac{\partial L}{\partial v_i} \cdot \frac{\partial v_i}{\partial v_j}$.

---

## Gradient Verification Against PyTorch

In [3]:
import torch

# Build the same expression: L = (a * b + c).tanh()
# Our engine
a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
L = (a * b + c).tanh()
L.backward()

# PyTorch
a_pt = torch.tensor(2.0, requires_grad=True)
b_pt = torch.tensor(-3.0, requires_grad=True)
c_pt = torch.tensor(10.0, requires_grad=True)
L_pt = (a_pt * b_pt + c_pt).tanh()
L_pt.backward()

print("Expression: L = tanh(a*b + c)  with a=2, b=-3, c=10")
print(f"{'':>8} {'Ours':>10} {'PyTorch':>10} {'Match':>8}")
for name, ours, theirs in [
    ('dL/da', a.grad, a_pt.grad.item()),
    ('dL/db', b.grad, b_pt.grad.item()),
    ('dL/dc', c.grad, c_pt.grad.item()),
]:
    match = abs(ours - theirs) < 1e-6
    print(f"{name:>8} {ours:>10.6f} {theirs:>10.6f} {'✓' if match else '✗':>8}")
    assert match, f"Gradient mismatch for {name}!"

print("\nAll gradients match PyTorch ✓")

Expression: L = tanh(a*b + c)  with a=2, b=-3, c=10
               Ours    PyTorch    Match
   dL/da  -0.004023  -0.004023        ✓
   dL/db   0.002682   0.002682        ✓
   dL/dc   0.001341   0.001341        ✓

All gradients match PyTorch ✓


---
**Understanding Reverse-Mode Autodiff and Topological Sort**

**Plain language:** Imagine you are at the end of a long chain of dominoes and you want to know how much wiggling the first domino would affect the last one. Instead of nudging each domino one at a time from the front (which would require a separate experiment per domino), you start from the last domino and ask each one in turn: "how much did your predecessor influence you?" Working backward gives you every domino's influence in a single sweep.

**Building intuition:** A topological sort of the computation graph lists nodes so that every node appears before any node that depends on it. The forward pass naturally follows this order — you cannot compute `a*b + c` before you know `a*b`. The backward pass reverses this order: you start from the output (where $\partial L / \partial L = 1$) and propagate gradients to predecessors. Processing in reverse-topological order guarantees that by the time you reach a node, all downstream gradient contributions have already been accumulated into its `.grad` field, so the local chain-rule multiplication uses the correct total gradient.

**Formal statement:** Let $v_1, v_2, \ldots, v_n$ be a topological ordering of the computation graph with $v_n = L$ (the loss). The backward pass iterates $i = n, n-1, \ldots, 1$. For each $v_i$ with children $\{v_k : (v_i, v_k) \in E\}$, the gradient is $\frac{\partial L}{\partial v_i} = \sum_{k} \frac{\partial L}{\partial v_k} \cdot \frac{\partial v_k}{\partial v_i}$. Because we process in reverse topological order, every $\frac{\partial L}{\partial v_k}$ is finalized before it is needed, and the total cost is $O(|E|)$ — one pass through all edges, regardless of the number of input parameters.

---

## Computation Graph Visualisation

In [4]:
def draw_graph(root, figsize=(12, 6)):
    """Visualise computation graph using matplotlib (no graphviz dependency)."""
    # Collect all nodes and edges via BFS
    nodes, edges = [], []
    visited = set()
    queue = [root]
    while queue:
        v = queue.pop(0)
        if id(v) in visited:
            continue
        visited.add(id(v))
        nodes.append(v)
        for child in v._prev:
            edges.append((child, v))
            queue.append(child)

    # Assign layers via reverse topological depth
    depth = {}
    def get_depth(v):
        if id(v) in depth:
            return depth[id(v)]
        if not v._prev:
            depth[id(v)] = 0
        else:
            depth[id(v)] = max(get_depth(c) for c in v._prev) + 1
        return depth[id(v)]

    for n in nodes:
        get_depth(n)

    # Position nodes
    max_d = max(depth.values())
    layers = {}
    for n in nodes:
        d = depth[id(n)]
        layers.setdefault(d, []).append(n)

    pos = {}
    for d, layer_nodes in layers.items():
        x = d / max(max_d, 1)
        for i, n in enumerate(layer_nodes):
            y = (i - (len(layer_nodes) - 1) / 2) * 0.3
            pos[id(n)] = (x, y)

    fig, ax = plt.subplots(figsize=figsize)
    # Draw edges
    for child, parent in edges:
        cx, cy = pos[id(child)]
        px, py = pos[id(parent)]
        ax.annotate('', xy=(px, py), xytext=(cx, cy),
                     arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

    # Draw nodes
    for n in nodes:
        x, y = pos[id(n)]
        label = n.label if n.label else n._op
        text = f"{label}\nval={n.data:.3f}\ngrad={n.grad:.3f}"
        bbox = dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.8)
        ax.text(x, y, text, ha='center', va='center', fontsize=8, bbox=bbox)

    ax.set_xlim(-0.15, 1.15)
    ax.axis('off')
    ax.set_title('Computation Graph (forward values + backward gradients)', fontweight='bold')
    plt.tight_layout()
    plt.show()

# Rebuild with labels for visualisation
a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
d = a * b; d.label = 'a*b'
e = d + c; e.label = 'a*b+c'
L = e.tanh(); L.label = 'L=tanh'
L.backward()

draw_graph(L)

/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_47336/285779051.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
def draw_forward_vs_backward(save_path='../assets/forward_vs_backward.png'):
    """Side-by-side forward values vs backward gradients for (a*b + c).tanh()."""

    # Build computation with fresh variables
    fv_a = Value(2.0, label='a')
    fv_b = Value(-3.0, label='b')
    fv_c = Value(10.0, label='c')
    fv_ab = fv_a * fv_b;  fv_ab.label = 'a*b'
    fv_sum = fv_ab + fv_c; fv_sum.label = 'a*b+c'
    fv_out = fv_sum.tanh(); fv_out.label = 'tanh'
    fv_out.backward()

    # Node layout (x, y) — manually positioned for clarity
    layout = {
        'a':     (0.0, 0.6),
        'b':     (0.0, 0.0),
        'c':     (0.0, -0.6),
        'a*b':   (0.35, 0.3),
        'a*b+c': (0.65, 0.0),
        'tanh':  (1.0, 0.0),
    }

    node_map = {
        'a': fv_a, 'b': fv_b, 'c': fv_c,
        'a*b': fv_ab, 'a*b+c': fv_sum, 'tanh': fv_out,
    }

    edge_list = [
        ('a', 'a*b'), ('b', 'a*b'),
        ('a*b', 'a*b+c'), ('c', 'a*b+c'),
        ('a*b+c', 'tanh'),
    ]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for panel_idx, (ax, mode) in enumerate(zip(axes, ['forward', 'backward'])):
        ax.set_xlim(-0.18, 1.18)
        ax.set_ylim(-1.0, 1.0)
        ax.axis('off')

        if mode == 'forward':
            ax.set_title('Forward Pass — Values', fontsize=14, fontweight='bold')
        else:
            ax.set_title('Backward Pass — Gradients', fontsize=14, fontweight='bold')

        # Draw edges
        for src, dst in edge_list:
            sx, sy = layout[src]
            dx, dy = layout[dst]
            if mode == 'backward':
                # Color by gradient magnitude of the destination node
                grad_mag = abs(node_map[dst].grad)
                intensity = min(grad_mag / 1.05, 1.0)  # normalize; tanh grad=1.0 is max
                edge_color = plt.cm.Reds(0.3 + 0.7 * intensity)
                # Arrows point backward (dst -> src)
                ax.annotate('', xy=(sx, sy), xytext=(dx, dy),
                            arrowprops=dict(arrowstyle='->', color=edge_color,
                                            lw=1.5 + 2.5 * intensity))
            else:
                ax.annotate('', xy=(dx, dy), xytext=(sx, sy),
                            arrowprops=dict(arrowstyle='->', color='steelblue', lw=1.8))

        # Draw nodes
        for name, (nx, ny) in layout.items():
            node = node_map[name]
            if mode == 'forward':
                box_text = f"{name}\nval = {node.data:.4f}"
                face_color = '#d0e8f2'
            else:
                box_text = f"{name}\ngrad = {node.grad:.6f}"
                grad_mag = abs(node.grad)
                intensity = min(grad_mag / 1.05, 1.0)
                face_color = plt.cm.Reds(0.15 + 0.45 * intensity)

            bbox_props = dict(boxstyle='round,pad=0.4', facecolor=face_color,
                              edgecolor='#444444', linewidth=1.2, alpha=0.92)
            ax.text(nx, ny, box_text, ha='center', va='center',
                    fontsize=9, fontfamily='monospace', bbox=bbox_props)

    plt.suptitle('Forward vs Backward Pass:  L = tanh(a*b + c)',
                 fontsize=13, fontweight='bold', y=0.02)
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"Saved to {save_path}")

draw_forward_vs_backward()

## Probing — Edge Cases

Test that the autograd handles repeated variable usage correctly (the += accumulation in backward).

In [5]:
# Edge case: variable used twice  →  gradient must accumulate
a = Value(3.0, label='a')
b = a + a  # da/dL should be 2, not 1
b.backward()
assert a.grad == 2.0, f"Expected grad=2.0, got {a.grad}"
print(f"a + a: grad(a) = {a.grad} ✓  (correctly accumulated)")

# Edge case: a * a (squaring)
a = Value(3.0, label='a')
c = a * a  # dc/da = 2a = 6
c.backward()
assert a.grad == 6.0, f"Expected grad=6.0, got {a.grad}"
print(f"a * a: grad(a) = {a.grad} ✓  (correctly 2a = 6)")

# More complex: verify against PyTorch for (a**2 + 3*a - 1).exp()
a = Value(1.5, label='a')
L = (a ** 2 + 3 * a - 1).exp()
L.backward()

a_pt = torch.tensor(1.5, requires_grad=True)
L_pt = (a_pt ** 2 + 3 * a_pt - 1).exp()
L_pt.backward()

assert abs(a.grad - a_pt.grad.item()) < 1e-4, f"Mismatch: {a.grad} vs {a_pt.grad.item()}"
print(f"(a² + 3a - 1).exp(): grad(a) = {a.grad:.4f} vs PyTorch {a_pt.grad.item():.4f} ✓")

a + a: grad(a) = 2.0 ✓  (correctly accumulated)
a * a: grad(a) = 6.0 ✓  (correctly 2a = 6)
(a² + 3a - 1).exp(): grad(a) = 1885.1440 vs PyTorch 1885.1440 ✓


---
**Understanding Gradient Accumulation (the += pattern)**

**Plain language:** Suppose you are a manager and two different teams both depend on the same supplier. If you want to know "how much does the supplier affect our total output?", you cannot just ask one team — you have to add up the influence through both teams. The `+=` in gradient computation does exactly this: it sums up every pathway through which a value influences the final result.

**Building intuition:** When a variable `a` appears multiple times in an expression (e.g., `a + a` or `a * a`), the computation graph has multiple edges leaving the node for `a`. Each edge represents a distinct path through which `a` influences the output. The chain rule for multivariable functions tells us to sum partial derivatives over all such paths. Using `=` instead of `+=` would overwrite the gradient from the first path when processing the second, silently producing wrong gradients. This is one of the most common bugs in hand-written autograd implementations.

**Formal statement:** For a node $v$ with multiple consumers $\{v_{k_1}, v_{k_2}, \ldots\}$ in the computation graph, the total derivative is given by the multivariate chain rule: $\frac{\partial L}{\partial v} = \sum_{j} \frac{\partial L}{\partial v_{k_j}} \cdot \frac{\partial v_{k_j}}{\partial v}$. The `+=` operator implements this summation incrementally as each consumer is processed during the backward pass. For example, in `a + a`, the gradient of `a` receives two contributions of $\frac{\partial L}{\partial \text{out}} \cdot 1$, yielding $\frac{\partial L}{\partial a} = 2 \cdot \frac{\partial L}{\partial \text{out}}$.

---

## Structured Interpretation

1. **Reverse-mode autodiff** computes all gradients in a single backward pass — O(1) passes regardless of the number of parameters. This is why it's preferred for neural networks (many parameters, scalar loss).

2. **Topological sort** ensures we process each node only after all its consumers, so gradients are fully accumulated before propagation.

3. **The += pattern** is essential: when a variable appears multiple times in an expression (e.g., `a + a` or `a * a`), its gradient must accumulate contributions from each usage path. This is the multivariate chain rule in action.

4. **Our Value class matches PyTorch to <1e-6** — the same mathematical operations underpin both, just at different scales (scalar vs tensor).

5. **Next notebook**: We'll use this Value class to build Neuron and Layer abstractions, constructing a trainable MLP from these scalar primitives.